#***Audio Feature Extraction***

In this step, we convert raw audio waveforms into compact, information-rich numerical representations. These features capture the temporal, spectral, and timbral characteristics of the sound, making them ideal for training machine learning models.

1. Mel-Frequency Cepstral Coefficients (MFCCs)

What it does: Represents the short-term power spectrum of a sound based on a linear cosine transform of a log power spectrum on a nonlinear mel scale of frequency.

Why we use it: It mimics human hearing perception, capturing crucial timbral characteristics. In our workflow, we typically extract the first 13 coefficients.   

2. Spectrograms & Mel-Spectrograms

Short-Time Fourier Transform (STFT): Computes the frequency content of local audio frames over time.  

Log-Amplitude Scaling: Converts the linear amplitude to decibels (amplitude_to_db). This visually highlights where signal energy is structurally conserved.  

Mel-Spectrogram: Warps the frequencies onto the Mel scale, providing an excellent visual feature map for 2D Convolutional Neural Networks (CNNs).

3. Additional Spectral Features

Chroma Feature: Projects the entire spectrum onto 12 bins representing the 12 distinct semitones (or chroma), making it highly effective for pitch and harmonic analysis.

Spectral Centroid: Indicates where the "center of mass" of the spectrum is located, closely correlating with the perceived "brightness" of a sound.

Import libraries:



In [ ]:
import librosa
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm           #used to generate progress bar for loops

Mounting google Drive to Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Defining Paths

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/Underwater Audio Data/augmented")

FEATURE_DIR = Path("/content/drive/MyDrive/Underwater Audio Data/features")
FEATURE_DIR.mkdir(exist_ok=True)

CSV_PATH = FEATURE_DIR / "audio_features.csv"

Setting up Parameters for Feature Extraction

In [ ]:
TARGET_SR = 16000

N_MFCC = 13

N_FFT = 2048

HOP_LENGTH = 512

N_MELS = 128

The feature Extraction function:

MFCC:-Mel Frequency Cepstral Coefficient.It is used to represent the distinct timbre of the sound.Mel frequency uses Mel scale which closely mimics how humans percieve sound.MFCCs strip away the background unwanted noises keeping only the core characteristics of sound.

Mel Spectogram:-A mel spectrogram is a visual representation of the frequency content of an audio signal over time, where the frequencies are warped to match human pitch perception(Mel frequency).t is created by applying the mel scale to a standard spectrogram.

Chroma:-It extracts a chromagram, a 12-element vector,representing the musical energy of the 12 semitones of the musical scale.

Chroma_stft: Computes a Short-Time Fourier Transform (STFT) and maps the linear frequency bins to chroma. It is fast and suitable for general-purpose analysis.

Spectral Centroid:-The spectral centroid is a measure in digital signal processing that indicates where the "center of mass" or "center of gravity" of a sound's frequency spectrum is located.Perceptually, the spectral centroid closely correlates with the "brightness" or "darkness" of a sound.

Zero CrossingRate:-The rate at which a digital or analog signal changes sign, meaning how many times its amplitude crosses the zero axis per unit time.




In [ ]:
def extract_features(y, sr):

    features = {}

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC
    )

    mfcc_mean = np.mean(mfcc, axis=1)

    for i, value in enumerate(mfcc_mean):
        features[f"mfcc_{i+1}"] = value

    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel)

    mel_mean = np.mean(mel_db, axis=1)

    for i, value in enumerate(mel_mean):
        features[f"mel_{i+1}"] = value

    # Chroma
    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    chroma_mean = np.mean(chroma, axis=1)

    for i, value in enumerate(chroma_mean):
        features[f"chroma_{i+1}"] = value

    # Spectral Centroid
    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )

    features["spectral_centroid"] = np.mean(centroid)

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y)

    features["zero_crossing_rate"] = np.mean(zcr)

    return features

Looping through all the audio files and extracting features from each of them. Each file has its own dictionary hlding all the features, label and file_name inside. A collection of this dictionary is stored in the "rows" list.


HERE THE recording_id IS VERY IMPORTANT FOR THE NEXT STEP OF TRAIN_TEST SPLIT. BECAUSE WE HAVE USED WINDOWING AND AUGMENTATION TECHNIQUES, TO PREVENT DATA LEAK, WE NEED TO MAKE SURE THAT FILES ORIGINATED FROM THE SAME ORIGINAL AUDIO FILE, BELONG IN THE SAME SPLIT. TO DO THIS WE NEED THE recording_id, WHICH IS BASICALLY THE NAME OF THE ORIGINAL AUDIO FILE.

In [ ]:
rows = []         #initialising an empty list which will hold a dictionary
                  #for each audio file, and each dictionary contains extracted features,
                  #the file name and corresponding label.


for label_dir in DATA_DIR.iterdir():

    if not label_dir.is_dir():
        continue

    label = label_dir.name

    wav_files = list(label_dir.glob("*.wav"))

    for wav_file in tqdm(wav_files):

        y, sr = librosa.load(
            wav_file,
            sr=None    #because already resampled to 16kHz
        )

        feature_dict = extract_features(y, sr)
        filename = wav_file.name

        recording_id=filename.split("_chunk")[0]

        feature_dict["recording_id"]=recording_id #HERE RRECORDING ID BECOMES VERY IMPORTANT

        feature_dict["filename"] = wav_file.name
        feature_dict["label"] = label

        rows.append(feature_dict)

Pandas convert the list of dictionaries into a structured table, where each dictionary becomes a row and the keys of the dictionaries become the coloumn header.

In [ ]:
df = pd.DataFrame(rows)

This code displays the first 5 rows of df dataframe.

In [ ]:
df.head()

In [ ]:
df["label"].value_counts()

It saves the df dataframe into a Comma seoarated values(CSV)file.

In [ ]:
df.to_csv(CSV_PATH, index=False)

print("Saved successfully!")

Now, i want to check for missing values. So ill have to import it from drive(the features csv file) and perform operation on it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Underwater Audio Data/features/audio_features.csv')

In [ ]:
print(df.isnull().sum())

Hence there are no missing values .

In [ ]:
filename_to_check = input("Enter the filename of the audio you want to check (e.g., 'icecalving_chunk16_pitch.wav'): ")

# Filter the DataFrame for the specified filename
file_features = df[df['filename'] == filename_to_check]

# Check if the file was found
if not file_features.empty:
    print(f"\nFeatures for {filename_to_check}:\n")
    # Display all columns except 'filename' and 'label'
    print(file_features.drop(columns=['filename', 'label']).iloc[0])
else:
    print(f"\nError: File '{filename_to_check}' not found in the features.csv.")